# Production Planning Assistant
**Manufacturing AI/ML Series - Playbook #2**

FlowForge Industries produces precision-engineered forgings. Production planners manually enter planned end dates when creating work orders - and they get them wrong. This notebook measures how wrong, identifies the root cause, and builds a calibration table that makes planning accurate.

**Data:** ForgeFlow ERP (fictional) - 50 work orders, 250 job cards, 190 stock entries, 115 sales orders

**Technique:** Planned vs actual lead time analysis, bottleneck identification, lead time calibration model

**Requirements:** No API key. No local setup. Runs entirely in Google Colab.

---
Reference platform: [ForgeFlow on GitHub](https://github.com/Nub-Labs/forgeflow-reference-platform)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

AMBER = '#D97706'
RED   = '#DC2626'
GREEN = '#16A34A'
GRAY  = '#6B7280'

In [ ]:
# Load ForgeFlow production and sales data from GitHub
BASE = 'https://raw.githubusercontent.com/Nub-Labs/forgeflow-reference-platform/main/data'

work_orders   = pd.read_json(f'{BASE}/production/work_orders.json')
job_cards     = pd.read_json(f'{BASE}/production/job_cards.json')
stock_entries = pd.read_json(f'{BASE}/production/stock_entries.json')
sales_orders  = pd.read_json(f'{BASE}/sales/sales_orders.json')

print(f'Work orders:   {len(work_orders):>4}')
print(f'Job cards:     {len(job_cards):>4}')
print(f'Stock entries: {len(stock_entries):>4}')
print(f'Sales orders:  {len(sales_orders):>4}')

In [ ]:
# Parse dates and compute planned vs actual lead times
date_cols = ['planned_start_date', 'planned_end_date', 'actual_start_date', 'actual_end_date']
for col in date_cols:
    work_orders[col] = pd.to_datetime(work_orders[col])

work_orders['planned_lead_days'] = (work_orders['planned_end_date'] - work_orders['planned_start_date']).dt.days
work_orders['actual_lead_days']  = (work_orders['actual_end_date']  - work_orders['actual_start_date']).dt.days
work_orders['overrun_days']      = work_orders['actual_lead_days'] - work_orders['planned_lead_days']
work_orders['on_time']           = work_orders['overrun_days'] <= 0
work_orders['item_family']       = work_orders['item_code'].str[:5]
work_orders['month']             = work_orders['actual_start_date'].dt.to_period('M')

late_count = (~work_orders['on_time']).sum()
avg_overrun = work_orders.loc[~work_orders['on_time'], 'overrun_days'].mean()

print(f'Work orders analysed: {len(work_orders)}')
print(f'Late (missed plan):   {late_count} ({late_count/len(work_orders)*100:.0f}%)')
print(f'Avg overrun (late):   {avg_overrun:.1f} days')

In [ ]:
# Chart 1: On-time delivery rate by item family
family_summary = work_orders.groupby(['item_family', 'item_name']).agg(
    total_wos=('wo_id', 'count'),
    on_time_count=('on_time', 'sum'),
    avg_planned=('planned_lead_days', 'mean'),
    avg_actual=('actual_lead_days', 'mean'),
    avg_overrun=('overrun_days', 'mean'),
).reset_index()
family_summary['on_time_pct'] = family_summary['on_time_count'] / family_summary['total_wos'] * 100
family_summary = family_summary.sort_values('on_time_pct')
family_summary['short_name'] = family_summary['item_name'].str.replace(' Class 300', '').str.replace(' Module 3', '').str.replace(' 4-Cyl', '').str.replace(' 3-Cyl', '')

fig, ax = plt.subplots(figsize=(10, 4))
colors = [GREEN if p == 100 else RED for p in family_summary['on_time_pct']]
bars = ax.barh(family_summary['short_name'], family_summary['on_time_pct'], color=colors, height=0.6)

for bar, val in zip(bars, family_summary['on_time_pct']):
    label = f'{val:.0f}%' if val > 0 else '0%'
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            label, va='center', fontsize=9, color='gray')

ax.set_xlim(0, 115)
ax.set_xlabel('On-time rate (%)')
ax.set_title('On-time delivery rate by item family\n7 of 9 item families: 0% on-time', pad=12)
ax.axvline(x=80, color='orange', linestyle='--', linewidth=0.8, label='80% target')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(f'Items with 100% on-time: {(family_summary["on_time_pct"]==100).sum()}')
print(f'Items with 0% on-time:   {(family_summary["on_time_pct"]==0).sum()}')

In [ ]:
# Chart 2: Planned vs actual lead time - the planning gap
fs_sorted = family_summary.sort_values('avg_overrun', ascending=True)
x = np.arange(len(fs_sorted))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 4.5))
bars_p = ax.bar(x - width/2, fs_sorted['avg_planned'], width, label='Planned days', color='#93C5FD', zorder=3)
bars_a = ax.bar(x + width/2, fs_sorted['avg_actual'],  width, label='Actual days',  color=AMBER,     zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(fs_sorted['short_name'], rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Lead time (days)')
ax.set_title('Planned vs actual lead time per item family\nEvery item takes ~18 days actual - planned estimates range 1-34 days', pad=12)
ax.axhline(y=18, color=GRAY, linestyle=':', linewidth=1, label='~18 day floor')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.set_ylim(0, 40)
plt.tight_layout()
plt.show()

print('Actual lead times per family:')
for _, row in fs_sorted.iterrows():
    print(f'  {row["item_family"]}: planned {row["avg_planned"]:.0f}d, actual {row["avg_actual"]:.0f}d, gap {row["avg_overrun"]:+.0f}d')

In [ ]:
# Chart 3: Workstation capacity allocation
ws_counts = job_cards.groupby('workstation').size().reset_index(name='job_count')
ws_counts['pct'] = ws_counts['job_count'] / ws_counts['job_count'].sum() * 100
ws_counts = ws_counts.sort_values('job_count', ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))
ws_colors = [RED if 'Forging' in w else AMBER if 'Furnace' in w else GRAY
             for w in ws_counts['workstation']]
bars = ax.barh(ws_counts['workstation'], ws_counts['job_count'], color=ws_colors, height=0.55)

for bar, row in zip(bars, ws_counts.itertuples()):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            f'{row.job_count} jobs ({row.pct:.0f}%)', va='center', fontsize=9)

ax.set_xlabel('Number of job cards')
ax.set_title('Job cards per workstation\nHeavy Forging Press handles 36% of all operations (Die Forging + Flash Trimming)', pad=12)
ax.set_xlim(0, 120)
plt.tight_layout()
plt.show()

total_jobs = ws_counts['job_count'].sum()
heavy_jobs = ws_counts.loc[ws_counts['workstation'] == 'Heavy Forging Press', 'job_count'].values[0]
print(f'Total job cards: {total_jobs}')
print(f'Heavy Forging Press: {heavy_jobs} jobs ({heavy_jobs/total_jobs*100:.0f}% of all capacity)')

In [ ]:
# Chart 4: Monthly production volume by item family (heatmap)
work_orders['month_str'] = work_orders['actual_start_date'].dt.strftime('%b %Y')
monthly = work_orders.groupby(['month_str', 'item_family']).size().unstack(fill_value=0)
monthly = monthly.reindex(sorted(monthly.index, key=lambda x: pd.to_datetime(x, format='%b %Y')))

fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(monthly.values.T, cmap='YlOrRd', aspect='auto')

ax.set_xticks(range(len(monthly.index)))
ax.set_xticklabels(monthly.index, rotation=0)
ax.set_yticks(range(len(monthly.columns)))
ax.set_yticklabels(monthly.columns)

for i in range(len(monthly.columns)):
    for j in range(len(monthly.index)):
        val = monthly.values[j, i]
        if val > 0:
            ax.text(j, i, str(val), ha='center', va='center', fontsize=9,
                    color='white' if val > 2 else 'black', fontweight='bold')

ax.set_title('Work orders started per item family per month\nProduction load distributed evenly across the 5-month period', pad=12)
plt.colorbar(im, ax=ax, shrink=0.6, label='WOs started')
plt.tight_layout()
plt.show()

In [ ]:
# Chart 5: Raw material consumption by material type
rm = stock_entries[stock_entries['line_type'] == 'rm_consumed'].copy()
rm_summary = rm.groupby(['item_code', 'item_name', 'uom']).agg(
    total_qty=('qty', 'sum'),
    total_amount=('amount', 'sum'),
    wo_count=('wo_id', 'nunique'),
).reset_index().sort_values('total_qty', ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
colors_rm = [AMBER if 'Billet' in n else GRAY for n in rm_summary['item_name']]
bars = ax.barh(rm_summary['item_name'].str.replace('Based Forging Lubricant', 'Lubricant'),
               rm_summary['total_qty'], color=colors_rm, height=0.55)

for bar, row in zip(bars, rm_summary.itertuples()):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height() / 2,
            f'{row.total_qty:,.0f} {row.uom} ({row.wo_count} WOs)', va='center', fontsize=8)

ax.set_xlabel('Total quantity consumed')
ax.set_title('Raw material consumption across all work orders\nEN8 Steel Billet (50mm) is the highest-volume input by weight', pad=12)
ax.set_xlim(0, 12500)
plt.tight_layout()
plt.show()

print('Top 3 materials by cost:')
for _, row in rm_summary.nlargest(3, 'total_amount').iterrows():
    print(f'  {row["item_name"]}: {row["total_amount"]:,.0f} INR ({row["wo_count"]} WOs)')

In [ ]:
# Lead time calibration model
# For each item family: compute recommended lead days from historical actuals + buffer

BUFFER_DAYS = 2  # 2-day safety buffer over the historical floor

calibration = family_summary[[
    'item_family', 'short_name', 'avg_planned', 'avg_actual', 'avg_overrun', 'on_time_pct'
]].copy()
calibration['recommended_days'] = np.ceil(calibration['avg_actual'] + BUFFER_DAYS).astype(int)
calibration['planning_status'] = calibration['avg_overrun'].apply(
    lambda x: 'UNDER-PLANNED' if x > 2 else ('OVER-BUFFERED' if x < -5 else 'ON-TRACK')
)
calibration = calibration.sort_values('avg_overrun', ascending=False)

print('Lead Time Calibration Model')
print('=' * 80)
print(f'{"Item":<12} {"Name":<26} {"Planned":>8} {"Actual":>7} {"Gap":>6} {"On-time":>8} {"Recommended":>12} {"Status"}')
print('-' * 80)
for _, row in calibration.iterrows():
    gap_str = f"{row['avg_overrun']:+.0f}d"
    print(f"{row['item_family']:<12} {row['short_name']:<26} {row['avg_planned']:>7.0f}d {row['avg_actual']:>7.0f}d {gap_str:>6} {row['on_time_pct']:>7.0f}% {row['recommended_days']:>11}d  {row['planning_status']}")
print('=' * 80)
print(f'\nRecommended standard lead time: {int(calibration["recommended_days"].median())} days (median) for all item families')
print(f'Current range: {calibration["avg_planned"].min():.0f}d to {calibration["avg_planned"].max():.0f}d (planned) vs {calibration["avg_actual"].min():.0f}d to {calibration["avg_actual"].max():.0f}d (actual)')

In [ ]:
# Sales order delivery risk analysis
# Which SOs have delivery_date commitments that conflict with realistic lead times?
sales_orders['order_date']    = pd.to_datetime(sales_orders['order_date'])
sales_orders['delivery_date'] = pd.to_datetime(sales_orders['delivery_date'])
sales_orders['so_lead_days']  = (sales_orders['delivery_date'] - sales_orders['order_date']).dt.days
sales_orders['item_family']   = sales_orders['item_code'].str[:5]

REALISTIC_LEAD = 20  # recommended lead time from calibration model
sales_orders['at_risk'] = sales_orders['so_lead_days'] < REALISTIC_LEAD

risk_summary = sales_orders.groupby('item_family').agg(
    total_sos=('so_id', 'count'),
    at_risk_count=('at_risk', 'sum'),
    avg_so_lead=('so_lead_days', 'mean'),
).reset_index()
risk_summary['at_risk_pct'] = risk_summary['at_risk_count'] / risk_summary['total_sos'] * 100

fig, ax = plt.subplots(figsize=(10, 4))
colors_risk = [RED if p > 50 else AMBER if p > 0 else GREEN for p in risk_summary['at_risk_pct']]
bars = ax.bar(risk_summary['item_family'], risk_summary['at_risk_pct'], color=colors_risk, zorder=3)
ax.axhline(y=50, color=GRAY, linestyle='--', linewidth=0.8)

for bar, row in zip(bars, risk_summary.itertuples()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{row.at_risk_count}/{row.total_sos}', ha='center', fontsize=8)

ax.set_ylabel('Sales orders at risk (%)')
ax.set_title(f'Sales orders with delivery window < {REALISTIC_LEAD} days (realistic production lead time)', pad=12)
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

total_at_risk = sales_orders['at_risk'].sum()
print(f'Sales orders with delivery window < {REALISTIC_LEAD} days: {total_at_risk} of {len(sales_orders)} ({total_at_risk/len(sales_orders)*100:.0f}%)')

In [ ]:
# Summary: What the production planning assistant does
print('Production Planning Assistant - Output Summary')
print('=' * 60)
print(f'Work orders analysed:        {len(work_orders)}')
print(f'Late work orders:            {(~work_orders["on_time"]).sum()} ({(~work_orders["on_time"]).mean()*100:.0f}%)')
print(f'Avg overrun (late WOs):      {work_orders.loc[~work_orders["on_time"], "overrun_days"].mean():.1f} days')
print()
print(f'Worst planning gap:')
worst = calibration.iloc[0]
print(f'  {worst["item_family"]} ({worst["short_name"]})')
print(f'  Planned: {worst["avg_planned"]:.0f}d  Actual: {worst["avg_actual"]:.0f}d  Gap: {worst["avg_overrun"]:+.0f}d')
print()
print(f'Key bottleneck: Heavy Forging Press (90 of 250 job cards - 36%)')
print(f'All items complete in 17-19 days actual (process-driven floor)')
print()
print(f'Calibrated lead time standard: 20 days for all item families')
print(f'  Reduces under-planning for 7 families')
print(f'  Recovers 16 days of wasted buffer for FG-KP')
print()
print(f'Sales at-risk: {total_at_risk} of {len(sales_orders)} SOs ({total_at_risk/len(sales_orders)*100:.0f}%) committed within < 20-day window')